# Notebook 3: Sparse Index + Crash Recovery (the BEST practices)

Notebook 2 gave us segments and cheap retention, but two real-world problems remain:

1. **Lookup is slow.** To find record #12345 we scan from the start of a segment.
2. **Crashes leave half-written records.** If the process dies mid-write, the tail of the active segment is garbage, and a naive reader will explode.

We fix both here.

## Setup

```bash
cd 02-distributed-primitives/segmented-log
uv sync
```

Select the `.venv` kernel in VS Code. If it doesn't appear, reload the window (`Cmd+Shift+P` -> **Reload Window**).

## Part 1: Sparse index

Kafka keeps an `.index` file next to each `.log` segment. It maps **some** logical offsets (every Nth record) to a byte position in the segment. This is a *sparse* index: small enough to fit in RAM, but good enough that a lookup does one seek + a short linear scan.

**Analogy:** a book has a table of contents (sparse - chapter granularity), not an index of every word. Finding a sentence = jump to chapter, then read a few pages.

```
offset 0   -> byte 0
offset 16  -> byte 240
offset 32  -> byte 512   <- to find offset 37, seek here, then read 5 records.
offset 48  -> byte 764
```

In [ ]:
import os, tempfile, glob

class IndexedSegmentedLog:
    """Segmented log + in-memory sparse index for O(log N) offset lookup."""

    INDEX_EVERY = 16  # record one entry every 16 records (tunable)

    def __init__(self, dir_, max_segment_bytes=1024):
        self.dir = dir_
        os.makedirs(dir_, exist_ok=True)
        self.max_bytes = max_segment_bytes
        self.active_id = 0
        self.next_offset = 0           # global logical offset across all segments
        # index[segment_id] = list of (logical_offset, byte_position)
        self.index = {}          # filled by append(); every INDEX_EVERY-th record
        self.seg_base = {0: 0}         # first logical offset of each segment
        self._open_active()

    def _seg_path(self, sid):
        return os.path.join(self.dir, f'segment-{sid:08d}.log')

    def _open_active(self):
        self.fp = open(self._seg_path(self.active_id), 'ab')
        self.active_size = self.fp.tell()

    def append(self, record: bytes):
        framed = len(record).to_bytes(4, 'big') + record
        if self.active_size + len(framed) > self.max_bytes and self.active_size > 0:
            self._roll()
        pos_in_seg = self.active_size
        self.fp.write(framed)
        self.active_size += len(framed)
        # Sparse index: only every Nth record goes in. Skip if this offset is
        # already the segment's anchor entry (added by _roll / the first append),
        # otherwise we'd store the same (offset, pos) pair twice.
        entries = self.index.setdefault(self.active_id, [(self.next_offset, pos_in_seg)])
        if self.next_offset % self.INDEX_EVERY == 0 and entries[-1][0] != self.next_offset:
            entries.append((self.next_offset, pos_in_seg))
        self.next_offset += 1

    def _roll(self):
        self.fp.close()
        self.active_id += 1
        self.seg_base[self.active_id] = self.next_offset
        # Every segment needs an anchor at byte 0, otherwise a lookup for an offset
        # below the segment's first indexed record has nowhere to start scanning.
        self.index[self.active_id] = [(self.next_offset, 0)]
        self._open_active()

    def read_at(self, offset: int) -> bytes:
        """Return the record at logical offset. Uses the sparse index."""
        if offset < 0 or offset >= self.next_offset:
            raise IndexError(offset)
        # 1. Find which segment holds this offset (the last segment whose base <= offset).
        sid = max(s for s, base in self.seg_base.items() if base <= offset)
        # 2. In that segment's index, find the largest entry with logical_offset <= offset.
        entries = self.index[sid]
        lo, hi = 0, len(entries) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if entries[mid][0] <= offset:
                lo = mid
            else:
                hi = mid - 1
        start_offset, start_pos = entries[lo]
        # 3. Seek there and scan forward (offset - start_offset) records.
        self.fp.flush()
        with open(self._seg_path(sid), 'rb') as f:
            f.seek(start_pos)
            skip = offset - start_offset
            for _ in range(skip):
                n = int.from_bytes(f.read(4), 'big')
                f.seek(n, 1)  # relative seek, skip the payload
            n = int.from_bytes(f.read(4), 'big')
            return f.read(n)

    def close(self):
        self.fp.close()

WORKDIR = tempfile.mkdtemp(prefix='seglog_idx_')
log = IndexedSegmentedLog(WORKDIR, max_segment_bytes=1024)
for i in range(500):
    log.append(f'event-{i:05d}'.encode())

print('segments written:', log.active_id + 1)
print('index entries per segment:', {sid: len(e) for sid, e in log.index.items()})

# The index must be sparse (that is the whole point) and strictly increasing.
total_entries = sum(len(e) for e in log.index.values())
assert total_entries < log.next_offset / 4, f'{total_entries} entries for 500 records is not sparse'
for sid, entries in log.index.items():
    offs = [o for o, _ in entries]
    assert offs == sorted(set(offs)), f'segment {sid} index has duplicate/unordered entries: {entries}'
print(f'✔ {total_entries} index entries cover {log.next_offset} records '
      f'({log.next_offset / total_entries:.0f} records per entry)')

### Lookups are now near-instant

No matter where the record is in the log, we do at most ONE seek + a short scan of `INDEX_EVERY - 1` records. Compare this with "open segment, read from byte 0 until you counted N records" - which is O(N).

In [ ]:
import time

def scan_length(log, offset):
    '''How many records we must skip after seeking. This is the cost the index buys down.'''
    sid = max(s for s, base in log.seg_base.items() if base <= offset)
    entries = log.index[sid]
    start = max(o for o, _ in entries if o <= offset)
    return offset - start

print(f"{'offset':>7} {'record':>14} {'records scanned':>16} {'time':>9}")
for target in [0, 7, 42, 123, 499]:
    t0 = time.perf_counter()
    rec = log.read_at(target)
    dt_us = (time.perf_counter() - t0) * 1e6
    print(f'{target:>7d} {rec.decode():>14} {scan_length(log, target):>16d} {dt_us:>7.1f} us')

# Correctness: every single offset resolves to the right record.
assert all(log.read_at(i) == f'event-{i:05d}'.encode() for i in range(log.next_offset))
# Bounded work: the index guarantees we never scan more than INDEX_EVERY-1 records,
# no matter how far into the log we look. That is the O(1)-after-seek property.
worst = max(scan_length(log, i) for i in range(log.next_offset))
assert worst < log.INDEX_EVERY, f'scanned {worst} records, index promised < {log.INDEX_EVERY}'
print(f'\n✔ 500 lookups all correct; worst case scanned {worst} records '
      f'(a full scan would have been up to {log.next_offset - 1})')

## Part 2: Crash recovery with a torn write

Appends are not atomic. A power loss or kill -9 in the middle of a write can leave:

- a length header but no payload, or
- a truncated payload.

A naive reader will try to read `length` bytes, hit EOF, and crash - or worse, silently return garbage. **Best practice:** on startup, scan the active segment and truncate back to the last *complete* record.

We do this in two steps: first with a length-only frame (which handles the *obvious* torn
write), then we show a torn write it silently accepts, and fix it with a **CRC** — which is
what Kafka and PostgreSQL actually store per record.

In [ ]:
import os, tempfile

# 1. Write a clean log of 5 records.
crash_dir = tempfile.mkdtemp(prefix='crash_')
seg = os.path.join(crash_dir, 'segment-00000000.log')
with open(seg, 'ab') as f:
    for i in range(5):
        r = f'msg-{i}'.encode()
        f.write(len(r).to_bytes(4, 'big'))
        f.write(r)
clean_size = os.path.getsize(seg)
print('clean segment size:', clean_size)

# 2. Simulate a crash mid-write: append a length header plus ONLY 2 of the 10 payload bytes.
with open(seg, 'ab') as f:
    f.write((10).to_bytes(4, 'big'))
    f.write(b'XX')
print('corrupted size:', os.path.getsize(seg))

In [ ]:
def recover(path: str) -> int:
    """Scan a segment, return the byte offset of the end of the last VALID record.

    Anything past that point is a torn write and should be truncated."""
    good_end = 0
    with open(path, 'rb') as f:
        while True:
            hdr = f.read(4)
            if len(hdr) < 4:
                break  # no room for another header -> stop
            n = int.from_bytes(hdr, 'big')
            payload = f.read(n)
            if len(payload) < n:
                break  # torn write: payload shorter than length said
            good_end = f.tell()
    return good_end

good = recover(seg)
print(f'last valid byte: {good} (file size was {os.path.getsize(seg)})')

# Truncate the corruption away. This is the SAFE recovery step.
with open(seg, 'r+b') as f:
    f.truncate(good)
print('after truncate, size:', os.path.getsize(seg))
assert os.path.getsize(seg) == clean_size
print('OK: segment recovered to last intact record')

### The torn write this does *not* catch

`recover()` trusts the length header. But a crash writes bytes, not records — the header
itself can be half-written, or the disk can hand back a plausible-looking header for a record
that was never committed. If those bytes happen to decode as a small length with that many
bytes behind them, recovery accepts a record the application never wrote and truncates
*after* it. Silent corruption, dressed up as successful recovery.

In [ ]:
# Rebuild a clean 5-record segment, then append bytes that *look* like a valid
# record: length=2, payload='XX'. Nothing ever wrote this; it is crash debris.
with open(seg, 'wb') as f:
    for i in range(5):
        r = f'msg-{i}'.encode()
        f.write(len(r).to_bytes(4, 'big')); f.write(r)
with open(seg, 'ab') as f:
    f.write((2).to_bytes(4, 'big') + b'XX')

good = recover(seg)
print(f'length-only recover() says everything up to byte {good} is valid '
      f'(file is {os.path.getsize(seg)} bytes)')

# It accepted the debris: good_end covers the phantom record, so truncation is a no-op
# and the next read returns a record the application never appended.
assert good == os.path.getsize(seg), 'expected the length-only scan to swallow the debris'
with open(seg, 'rb') as f:
    blob = f.read()
recs, pos = [], 0
while pos < len(blob):
    n = int.from_bytes(blob[pos:pos + 4], 'big'); pos += 4
    recs.append(blob[pos:pos + n]); pos += n
assert recs[-1] == b'XX', recs
print(f'💥 recovered records: {recs}  <- the last one is garbage')

### Fix: a CRC per record

Add a 4-byte CRC32 of the payload to the frame. Now a record is only accepted if the bytes
hash to what the writer recorded, so crash debris and bit-rot both fail the check and get
truncated away.

A CRC catches *accidental* damage. It is **not** a signature — anyone who can rewrite the
payload can recompute the CRC. See the `checksum` lab for where that line is.

In [ ]:
import struct, zlib

CRC_FRAME = struct.Struct('>II')   # length, crc32

def append_crc(path, record: bytes):
    with open(path, 'ab') as f:
        f.write(CRC_FRAME.pack(len(record), zlib.crc32(record)) + record)

def recover_crc(path) -> int:
    '''Byte offset of the end of the last record that is BOTH complete and intact.'''
    good_end = 0
    with open(path, 'rb') as f:
        while True:
            hdr = f.read(CRC_FRAME.size)
            if len(hdr) < CRC_FRAME.size:
                break                                  # torn header
            n, crc = CRC_FRAME.unpack(hdr)
            payload = f.read(n)
            if len(payload) < n or zlib.crc32(payload) != crc:
                break                                  # torn or corrupt payload
            good_end = f.tell()
    return good_end

seg2 = os.path.join(crash_dir, 'segment-00000001.log')
for i in range(5):
    append_crc(seg2, f'msg-{i}'.encode())
clean2 = os.path.getsize(seg2)

# 1. The same crash debris as before: a plausible header with bytes behind it.
with open(seg2, 'ab') as f:
    f.write(CRC_FRAME.pack(2, 0x1234) + b'XX')
assert recover_crc(seg2) == clean2, 'CRC should have rejected the phantom record'

# 2. A genuinely torn payload.
with open(seg2, 'r+b') as f:
    f.truncate(clean2)
with open(seg2, 'ab') as f:
    f.write(CRC_FRAME.pack(10, zlib.crc32(b'0123456789')) + b'01')
assert recover_crc(seg2) == clean2

# 3. Bit-rot inside a record that is the right length.
with open(seg2, 'r+b') as f:
    f.truncate(clean2)
raw = bytearray(open(seg2, 'rb').read())
raw[-1] ^= 0x01
open(seg2, 'wb').write(raw)
good2 = recover_crc(seg2)
assert good2 == clean2 - (CRC_FRAME.size + len(b'msg-4')), good2

print('✅ CRC framing rejected all three: phantom record, torn payload, and a flipped bit')
print(f'   (bit-rot case truncated back to byte {good2}, dropping only the damaged record)')

## Recap: the progression

| Step | What changed | Why it matters |
|------|--------------|----------------|
| NB1  | One giant file | Simple, but deletes rewrite everything (O(n)) |
| NB2  | Split into segments + roll | Retention becomes `unlink` (O(1) per segment) |
| NB3  | Sparse index + CRC-framed recovery | Bounded-scan lookups; a torn or corrupt tail is truncated, not replayed |

## Best practices you now know

- **Roll by size OR time**, whichever trips first. This caps segment size *and* freshness.
- **Zero-pad segment IDs** (`segment-00000042.log`) so directory listing order == log order.
- Keep an **in-memory sparse index**; persist a snapshot of it next to each closed segment so you don't have to rebuild on startup.
- **Length + CRC per record.** Length alone lets crash debris masquerade as a record; the CRC is what makes recovery trustworthy.
- On startup, **truncate the active segment** back to the last valid record.
- Only the **active** segment is mutable; older segments are immutable -> safe to share, memory-map, compress, or ship to S3.

## Further reading

- [Kafka: a distributed messaging system for log processing](https://notes.stephenholiday.com/Kafka.pdf) - the paper.
- [Kafka docs - log segments & index files](https://kafka.apache.org/documentation/#log).
- [PostgreSQL WAL internals](https://www.postgresql.org/docs/current/wal-internals.html).
- [Designing Data-Intensive Applications, Chapter 3](https://dataintensive.net/) - log-structured storage.
